# Hypergraph GNN + MILP: Weighted Set Cover

Set Cover is naturally a hypergraph problem:

- **element = node**,
- **candidate set = hyperedge**.

The objective is to select a minimum-cost collection of sets that covers every element.

This notebook implements:

```text
full Set Cover MILP
 -> sets selected by the optimum = training labels
 -> HypergraphConv
 -> set scores
 -> candidate-set screening
 -> coverage repair
 -> reduced MILP
```

The GNN does not replace the optimizer; it reduces the candidate decision space.


In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import milp, LinearConstraint, Bounds
from torch_geometric.nn import HypergraphConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Generate weighted Set Cover instances

`A[i,j] = 1` means that candidate set `j` covers element `i`.


In [ ]:
def generate_instance(n_elements=14, n_sets=22, density=0.28, seed=0):
    rng = np.random.default_rng(seed)
    A = rng.random((n_elements, n_sets)) < density

    # Ensure every element is covered by at least two candidate sets.
    for i in range(n_elements):
        idx = np.flatnonzero(A[i])
        if len(idx) < 2:
            missing = 2 - len(idx)
            candidates = np.setdiff1d(np.arange(n_sets), idx)
            A[i, rng.choice(candidates, size=missing, replace=False)] = True

    # Avoid empty candidate sets.
    for j in range(n_sets):
        if not A[:, j].any():
            A[rng.integers(n_elements), j] = True

    coverage = A.sum(axis=0)
    cost = (
        rng.uniform(1.0, 5.0, n_sets)
        + 0.65 * coverage
        + rng.uniform(0.0, 2.0, n_sets)
    )

    return {
        "A": A.astype(float),
        "c": cost.astype(float),
    }


example = generate_instance(seed=SEED)
print("elements:", example["A"].shape[0], "sets:", example["A"].shape[1])


## 2. Solve the full Set Cover MILP

\[
\min \sum_j c_j x_j
\]

subject to

\[
\sum_j A_{ij}x_j \ge 1 \qquad \forall i
\]

and

\[
x_j\in\{0,1\}.
\]


In [ ]:
def solve_set_cover(inst, allowed=None, time_limit=20.0):
    A, c = inst["A"], inst["c"]
    n_elements, n_sets = A.shape

    if allowed is None:
        allowed = np.arange(n_sets, dtype=int)
    else:
        allowed = np.array(sorted(set(map(int, allowed))), dtype=int)

    if len(allowed) == 0:
        return None

    # Immediate structural infeasibility check.
    if np.any(A[:, allowed].sum(axis=1) == 0):
        return None

    constraint = LinearConstraint(
        A[:, allowed],
        np.ones(n_elements),
        np.full(n_elements, np.inf),
    )

    t0 = time.perf_counter()
    res = milp(
        c=c[allowed],
        integrality=np.ones(len(allowed)),
        bounds=Bounds(
            np.zeros(len(allowed)),
            np.ones(len(allowed)),
        ),
        constraints=constraint,
        options={"time_limit": time_limit},
    )
    elapsed = time.perf_counter() - t0

    if not res.success or res.x is None:
        return None

    x = np.zeros(n_sets)
    x[allowed] = np.rint(res.x)

    return {
        "x": x,
        "obj": float(c @ x),
        "time": elapsed,
    }


full_example = solve_set_cover(example)
print("objective:", full_example["obj"])
print("selected sets:", np.flatnonzero(full_example["x"] > 0.5))


## 3. Hypergraph representation

`hyperedge_index[0]` stores the element/node ID and `hyperedge_index[1]` stores the set/hyperedge ID.

Node features:

- normalized element degree,
- minimum normalized cost among covering sets,
- average normalized cost among covering sets.

Hyperedge/set features:

- normalized set cost,
- coverage ratio,
- normalized cost per covered element.


In [ ]:
def build_graph(inst, label=None):
    A, c = inst["A"], inst["c"]
    n_elements, n_sets = A.shape

    rows, cols = np.nonzero(A)
    hyperedge_index = torch.tensor(
        np.vstack([rows, cols]),
        dtype=torch.long,
    )

    c_norm = c / max(c.max(), 1e-9)
    degree = A.sum(axis=1) / max(n_sets, 1)

    min_cost = []
    mean_cost = []

    for i in range(n_elements):
        idx = np.flatnonzero(A[i])
        min_cost.append(c_norm[idx].min())
        mean_cost.append(c_norm[idx].mean())

    node_x = np.column_stack([
        degree,
        min_cost,
        mean_cost,
    ]).astype(np.float32)

    coverage = A.sum(axis=0)
    set_x = np.column_stack([
        c_norm,
        coverage / max(n_elements, 1),
        c_norm / np.maximum(coverage, 1),
    ]).astype(np.float32)

    graph = {
        "node_x": torch.tensor(node_x),
        "set_x": torch.tensor(set_x),
        "hyperedge_index": hyperedge_index,
    }

    if label is not None:
        graph["y"] = torch.tensor(label, dtype=torch.float32)

    return graph


graph = build_graph(example, full_example["x"])
graph


## 4. Hypergraph model and a non-graph MLP baseline

`HypergraphConv` propagates through the incidence structure. After node updates, we aggregate updated element embeddings into one representation per candidate set/hyperedge.

The `SetMLP` baseline uses only set-level engineered features and ignores the incidence structure.


In [ ]:
def hyperedge_mean(node_embeddings, hyperedge_index, n_hyperedges):
    node_idx, edge_idx = hyperedge_index

    out = node_embeddings.new_zeros(
        (n_hyperedges, node_embeddings.size(1))
    )
    out.index_add_(0, edge_idx, node_embeddings[node_idx])

    count = node_embeddings.new_zeros((n_hyperedges, 1))
    count.index_add_(
        0,
        edge_idx,
        node_embeddings.new_ones((len(edge_idx), 1)),
    )

    return out / count.clamp_min(1.0)


class HyperSetCover(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.node_encoder = nn.Linear(3, hidden)
        self.conv1 = HypergraphConv(hidden, hidden)
        self.conv2 = HypergraphConv(hidden, hidden)
        self.head = nn.Sequential(
            nn.Linear(hidden + 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, g):
        h = F.relu(self.node_encoder(g["node_x"]))
        h = F.relu(self.conv1(h, g["hyperedge_index"]))
        h = F.relu(self.conv2(h, g["hyperedge_index"]))

        edge_h = hyperedge_mean(
            h,
            g["hyperedge_index"],
            g["set_x"].size(0),
        )

        return self.head(
            torch.cat([edge_h, g["set_x"]], dim=1)
        ).squeeze(1)


class SetMLP(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, g):
        return self.net(g["set_x"]).squeeze(1)


## 5. Build exact-label training data


In [ ]:
def make_dataset(count, start_seed):
    out = []
    seed = start_seed

    while len(out) < count:
        inst = generate_instance(seed=seed)
        sol = solve_set_cover(inst)
        seed += 1

        if sol is None:
            continue

        out.append((
            inst,
            sol,
            build_graph(inst, sol["x"]),
        ))

    return out


train_data = make_dataset(90, 1000)
test_data = make_dataset(30, 5000)

print("train:", len(train_data), "test:", len(test_data))


## 6. Train both models


In [ ]:
def to_device(g):
    return {k: v.to(device) for k, v in g.items()}


def train_model(model, epochs=70):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

    positives = sum(float(g["y"].sum()) for _, _, g in train_data)
    total = sum(g["y"].numel() for _, _, g in train_data)
    pos_weight = torch.tensor(
        max((total - positives) / max(positives, 1.0), 1.0),
        device=device,
    )

    for epoch in range(epochs):
        for idx in np.random.permutation(len(train_data)):
            g = to_device(train_data[idx][2])

            loss = F.binary_cross_entropy_with_logits(
                model(g),
                g["y"],
                pos_weight=pos_weight,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(
                model.__class__.__name__,
                "epoch",
                epoch + 1,
            )

    return model


hyper_model = train_model(HyperSetCover())
mlp_model = train_model(SetMLP())


## 7. Candidate-set screening and coverage repair

Screening may remove every covering set for an element. Before solving the reduced MILP, a simple repair restores the highest-scoring candidate for each uncovered element.


In [ ]:
@torch.no_grad()
def predict_scores(model, g):
    model.eval()
    return torch.sigmoid(model(to_device(g))).cpu().numpy()


def repair_coverage(inst, chosen, scores):
    A = inst["A"]
    chosen = set(map(int, chosen))

    for i in range(A.shape[0]):
        candidates = np.flatnonzero(A[i])

        if not any(j in chosen for j in candidates):
            best = candidates[np.argmax(scores[candidates])]
            chosen.add(int(best))

    return np.array(sorted(chosen), dtype=int)


def select_candidates(inst, scores, fraction=0.45):
    k = max(1, int(np.ceil(len(scores) * fraction)))
    chosen = np.argsort(scores)[-k:]
    return repair_coverage(inst, chosen, scores)


def cost_coverage_scores(inst):
    coverage = inst["A"].sum(axis=0)
    return coverage / np.maximum(inst["c"], 1e-9)


## 8. Evaluate the end-to-end OR metrics


In [ ]:
def evaluate(model=None, heuristic=False):
    rows = []

    for inst, full, g in test_data:
        if heuristic:
            scores = cost_coverage_scores(inst)
        else:
            scores = predict_scores(model, g)

        keep = select_candidates(inst, scores)
        reduced = solve_set_cover(inst, keep)

        optimal_sets = set(np.flatnonzero(full["x"] > 0.5).tolist())
        kept = set(keep.tolist())

        rows.append({
            "feasible": reduced is not None,
            "recall": len(optimal_sets & kept) / max(len(optimal_sets), 1),
            "retained": len(keep) / len(inst["c"]),
            "gap_pct": np.nan if reduced is None else (
                100.0 * (reduced["obj"] - full["obj"])
                / max(abs(full["obj"]), 1e-9)
            ),
            "full_time": full["time"],
            "reduced_time": np.nan if reduced is None else reduced["time"],
        })

    return rows


def summarize(name, rows):
    arr = lambda key: np.array([r[key] for r in rows], dtype=float)

    return {
        "method": name,
        "feasibility": arr("feasible").mean(),
        "optimal_set_recall": arr("recall").mean(),
        "retained_ratio": arr("retained").mean(),
        "mean_gap_pct": np.nanmean(arr("gap_pct")),
        "full_time": arr("full_time").mean(),
        "reduced_time": np.nanmean(arr("reduced_time")),
    }


results = [
    summarize("Hypergraph GNN", evaluate(hyper_model)),
    summarize("Set-feature MLP", evaluate(mlp_model)),
    summarize(
        "Cost/coverage heuristic",
        evaluate(heuristic=True),
    ),
]
results


## 9. Why this is genuinely a hypergraph problem

The Set Cover incidence matrix \(H\) directly represents `element i ∈ set j`.

A hypergraph convolution can be interpreted through node-to-hyperedge-to-node propagation such as

\[
X' = D^{-1} H W B^{-1} H^\top X\Theta.
\]

The hypergraph is therefore not decorative terminology. A single decision variable \(x_j\) simultaneously covers a group of elements, which is exactly the semantic role of a hyperedge.

A production-oriented pattern is:

```text
Hypergraph GNN
 -> candidate screening
 -> coverage repair
 -> MILP / CP solver
 -> validated solution
```

Serious experiments should also test distribution shift, stronger greedy/LP-rounding baselines, larger instances, exact-label generation cost, and complete end-to-end runtime.
